# 📊 Txoko — Pipeline de Datos: Open Data Euskadi
### Reto Inetum · Bootcamp BBK The Bridge · Equipo 4
**Autores:** Andoni (Data Lead), Naia, Unai, Fátima  
**Fecha:** Mayo 2026  
**Audiencia:** Equipo interno · Profesores · Jurado · Empresa Inetum

---

> **Principio de trabajo:** cada decisión técnica está documentada con (1) de dónde viene el dato, (2) por qué se hace así, (3) cómo se procesa, (4) qué limitaciones tiene.

---

## Índice
1. [Contexto y problema de datos](#1-contexto)
2. [Descubrimiento de fuentes — Open Data Euskadi](#2-descubrimiento)
3. [Descarga de datasets](#3-descarga)
4. [Auditoría de campos](#4-auditoria)
5. [Merge y normalización](#5-merge)
6. [Limpieza y calidad](#6-limpieza)
7. [Dataset maestro final](#7-final)


## 1. Contexto y problema de datos <a name="1-contexto"></a>

**¿Qué estamos construyendo?**  
Txoko es una app de descubrimiento auténtico de Bilbao/Euskadi. El usuario (persona: Mark/Elena, turista que quiere vivir como bilbaíno) puede explorar puntos de interés clasificados por categorías: Culinario, Cultural, Naturaleza, Ocio, Compras y Alojamiento.

**¿Por qué necesitamos datos estructurados?**  
La app necesita una base de datos de lugares con:
- Nombre, descripción, categoría
- Coordenadas (lat/lon) para el mapa
- Municipio y territorio (para filtrar por zona)
- Información de contacto (teléfono, web)

**¿De dónde vienen los datos?**  
Fuente única: **Open Data Euskadi** — el portal de datos abiertos del Gobierno Vasco.  
URL: https://opendata.euskadi.eus  
Licencia: Creative Commons Attribution 4.0 — uso libre con atribución.

**¿Por qué Open Data Euskadi y no Google Places, Foursquare, etc.?**  
1. **Gratuito y abierto** — sin límites de API ni coste
2. **Oficial** — datos publicados por el Gobierno Vasco, calidad garantizada
3. **Licencia libre** — podemos usarlos en un producto sin restricciones
4. **Específico de Euskadi** — incluye recursos que no aparecen en plataformas globales (sidrerías, caseríos, puertos pesqueros, etxades rurales)

**Limitación importante (honestidad ante el jurado):**  
Open Data Euskadi NO tiene eventos en tiempo real (conciertos, estropadak, festivos). Para esa categoría, en producción se conectaría a la API de la agenda de Bilbao o Eventbrite.


## Librerías necesarias

In [4]:
import urllib.request
import json
import re
import os
import pandas as pd
import numpy as np
import ssl
import certifi
from collections import Counter, defaultdict

# Directorios
RAW_DIR    = '../data/raw'  # JSONs descargados de Open Data Euskadi
OUTPUT_DIR = '../data'  # Dataset maestro procesado

print("✓ Librerías cargadas")
print(f"  pandas {pd.__version__} | numpy {np.__version__}")


✓ Librerías cargadas
  pandas 3.0.3 | numpy 2.4.6


## 2. Descubrimiento de fuentes <a name="2-descubrimiento"></a>

### El problema inicial

Open Data Euskadi **no tiene una API REST estándar** ni un índice descargable de todos sus datasets. Las URLs de los archivos JSON no son predecibles — cada dataset tiene un subdirectorio propio con un nombre arbitrario.

**Primer intento fallido:** construir URLs "a ciegas" como:
```
https://opendata.euskadi.eus/contenidos/ds_recursos_turisticos/hoteles/opendata/hoteles.json
```
→ HTTP 404. El subdirectorio real resultó ser `hoteles_de_euskadi`, no `hoteles`.

### La solución: el catálogo de datasets

El catálogo de datasets sí es navegable por tipo. La query que devuelve **todos los datasets turísticos** del Gobierno Vasco es:

```
https://opendata.euskadi.eus/catalogo-datos/?r01kQry=tC:euskadi;tT:ds_recursos_turisticos;m:documentLanguage.EQ.es;pp:r01PageSize.100
```

Esta query devolvió **50 datasets** del tipo `ds_recursos_turisticos`.

Cada dataset tiene su propia página de catálogo con el link de descarga real. El patrón es:
```
https://opendata.euskadi.eus/catalogo/-/NOMBRE-DEL-DATASET/
```

**Clave aportada por Unai (compañero de equipo):** los links directos a páginas de datasets específicos como:
- https://opendata.euskadi.eus/catalogo/-/playas-de-euskadi/
- https://opendata.euskadi.eus/catalogo/-/patrimonio-cultural-edificios-religiosos-castillos-y-estructuras-de-interes/

Esto permitió confirmar el patrón y encontrar las URLs reales de descarga.


In [6]:
# ─── PASO 1: Obtener lista de datasets del catálogo ────────────────────────
# 
# Hacemos una petición a la query del catálogo de Open Data Euskadi
# que filtra por tipo ds_recursos_turisticos.
# Extraemos con regex todos los slugs /catalogo/-/NOMBRE/ que aparecen en el HTML.

CATALOG_QUERY_URL = (
    "https://opendata.euskadi.eus/catalogo-datos/"
    "?r01kQry=tC:euskadi;tT:ds_recursos_turisticos;m:documentLanguage.EQ.es;pp:r01PageSize.100"
)

context = ssl.create_default_context(cafile=certifi.where())
req = urllib.request.Request(CATALOG_QUERY_URL, headers={'User-Agent': 'Mozilla/5.0'})

with urllib.request.urlopen(req, context=context, timeout=15) as r:
    html = r.read().decode('latin-1', errors='replace')

# Extraer todos los slugs de datasets
slugs = list(set(re.findall(r'/catalogo/-/([a-z0-9-]+)/', html)))
print(f"Datasets turísticos encontrados en el catálogo: {len(slugs)}")
for s in sorted(slugs):
    print(f"  /catalogo/-/{s}/")


Datasets turísticos encontrados en el catálogo: 50
  /catalogo/-/agencias-de-viaje-de-euskadi/
  /catalogo/-/albergues-turisticos-de-euskadi/
  /catalogo/-/alojamientos/
  /catalogo/-/alojamientos-rurales-de-euskadi/
  /catalogo/-/aquariums-de-euskadi/
  /catalogo/-/auditorios-de-euskadi/
  /catalogo/-/bares-de-pintxos-de-euskadi/
  /catalogo/-/campings-de-euskadi/
  /catalogo/-/campos-de-golf-de-euskadi/
  /catalogo/-/casinos-de-euskadi/
  /catalogo/-/centros-btt-de-euskadi-bicicleta-de-montana/
  /catalogo/-/comarcas-de-euskadi/
  /catalogo/-/companias-de-transporte-de-euskadi/
  /catalogo/-/destinos-turisticos-de-euskadi/
  /catalogo/-/empresas-de-alquiler-deportivo-de-euskadi/
  /catalogo/-/empresas-de-turismo-activo-de-euskadi/
  /catalogo/-/espacios-naturales-y-playas-de-euskadi/
  /catalogo/-/estaciones-de-transporte-de-euskadi/
  /catalogo/-/experiencias-y-planes-para-hacer-en-euskadi/
  /catalogo/-/gastronomia-de-euskadi/
  /catalogo/-/hipodromo-y-estadios-de-futbol-de-las-tre

### Extraer la URL de descarga real de cada dataset

Cada página de catálogo contiene un botón de descarga con la URL real del JSON.  
Iteramos sobre todos los slugs y extraemos el primer link `.json` que aparece en el HTML.

**¿Por qué JSON y no CSV o XLSX?**  
- El JSON contiene la estructura original del objeto, incluyendo campos anidados (como `address` con subcampos).
- El CSV de Open Data Euskadi aplana los campos de forma inconsistente entre datasets.
- Preferimos controlar nosotros el proceso de aplanado para garantizar consistencia.


In [16]:
# ─── PASO 2: Extraer URL real de descarga de cada dataset ──────────────────
#
# Para cada slug, accedemos a su página de catálogo y extraemos la URL del JSON.
# Usamos regex para buscar el patrón:
#   https://opendata.euskadi.eus/contenidos/.../*.json
#
# Resultado: diccionario {slug: url_json}

import time

dataset_urls = {}

for slug in sorted(slugs):
    url = f"https://opendata.euskadi.eus/catalogo/-/{slug}/"
    print(url)
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, context=context, timeout=8) as r:
            html_page = r.read().decode('latin-1', errors='replace')
        
        json_urls = re.findall(
            r'https?://opendata\.euskadi\.eus/contenidos/[^\s\"\'<>]+\.json',
            html_page
        )
        if json_urls:
            dataset_urls[slug] = json_urls[0]
            print(f"✓ {slug}")
            print(f"    → {json_urls[0].split('/contenidos/')[1]}")
        else:
            print(f"- {slug}: página existe pero sin JSON descargable")
    except Exception as e:
        print(f"✗ {slug}: {type(e).__name__}")
    
    time.sleep(0.1)  # cortesía al servidor

print(f"\n{'='*60}")
print(f"URLs de descarga encontradas: {len(dataset_urls)} / {len(slugs)}")


https://opendata.euskadi.eus/catalogo/-/agencias-de-viaje-de-euskadi/
✓ agencias-de-viaje-de-euskadi
    → ds_recursos_turisticos/agendas_viaje_euskadi/opendata/agencias_viaje.json
https://opendata.euskadi.eus/catalogo/-/albergues-turisticos-de-euskadi/
✓ albergues-turisticos-de-euskadi
    → ds_recursos_turisticos/albergues_de_euskadi/opendata/alojamientos.json
https://opendata.euskadi.eus/catalogo/-/alojamientos/
✓ alojamientos
    → ds_recursos_turisticos/alojamiento_de_euskadi/opendata/alojamientos.json
https://opendata.euskadi.eus/catalogo/-/alojamientos-rurales-de-euskadi/
✓ alojamientos-rurales-de-euskadi
    → ds_recursos_turisticos/alojamientos_rurales_euskadi/opendata/alojamientos.json
https://opendata.euskadi.eus/catalogo/-/aquariums-de-euskadi/
✓ aquariums-de-euskadi
    → ds_recursos_turisticos/aquariums_de_euskadi/opendata/aquariums.json
https://opendata.euskadi.eus/catalogo/-/auditorios-de-euskadi/
✓ auditorios-de-euskadi
    → ds_recursos_turisticos/auditorios_euskadi/o

## 3. Descarga de datasets <a name="3-descarga"></a>

### ¿Qué descargamos exactamente?

Para cada URL encontrada en el paso anterior:
1. Realizamos una petición HTTP GET
2. Verificamos que la respuesta es JSON válido (empieza por `[` o `{`)
3. Guardamos el archivo en `data/raw/NOMBRE.json`

**Verificación de integridad:** si la respuesta contiene HTML (página de error) en vez de JSON, la descartamos. Open Data Euskadi devuelve a veces HTML con mensaje de error en vez de un 404 limpio.

### Resultado de la descarga

De los ~50 datasets en el catálogo, **39 descargaron correctamente** con datos reales.  
Los restantes devolvieron respuestas vacías o HTML de error (datasets sin contenido publicado).


In [17]:
# ─── PASO 3: Descargar cada JSON y guardar en data/raw/ ─────────────────────
#
# Para cada URL, verificamos:
#   (a) que la respuesta es JSON (empieza por '[' o '{')
#   (b) que tiene contenido (más de 50 bytes)
# 
# Si pasa la verificación → guardamos como data/raw/NOMBRE.json
# Si no → registramos el fallo con el motivo

download_results = {}

for slug, url in dataset_urls.items():
    # Convertir slug a nombre de archivo (guiones → guiones bajos)
    fname = slug.replace('-', '_')
    # Simplificar nombres largos
    fname = re.sub(r'_de_euskadi$', '', fname)
    fname = re.sub(r'_de_las_tres_capitales.*', '', fname)
    
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, context=context, timeout=10) as r:
            data = r.read()
        
        # Verificar que es JSON válido
        if data[:1] not in [b'[', b'{']:
            download_results[fname] = {
                'status': 'ERROR', 
                'motivo': f'No es JSON, empieza por: {data[:30]}'
            }
            print(f"✗ {fname}: respuesta no es JSON")
            continue
        
        # Parsear para contar registros
        parsed = json.loads(data)
        n_records = len(parsed) if isinstance(parsed, list) else 1
        
        # Guardar
        filepath = os.path.join(RAW_DIR, f'{fname}.json')
        with open(filepath, 'wb') as f:
            f.write(data)
        
        download_results[fname] = {
            'status': 'OK',
            'n_registros': n_records,
            'size_kb': len(data) // 1024,
            'url_fuente': url
        }
        print(f"✓ {fname:45s} {n_records:4d} registros  {len(data)//1024}KB")
        
    except Exception as e:
        download_results[fname] = {'status': 'ERROR', 'motivo': str(e)}
        print(f"✗ {fname}: {e}")

ok = sum(1 for v in download_results.values() if v['status'] == 'OK')
total_records = sum(v.get('n_registros', 0) for v in download_results.values())
print(f"\n{'='*60}")
print(f"Descargados: {ok} / {len(download_results)} datasets")
print(f"Total registros: {total_records:,}")


✓ agencias_de_viaje                              532 registros  886KB
✓ albergues_turisticos                            82 registros  150KB
✓ alojamientos                                  1453 registros  2646KB
✓ alojamientos_rurales                           433 registros  678KB
✓ aquariums                                        1 registros  1KB
✓ auditorios                                       2 registros  3KB
✓ bares_de_pintxos                               132 registros  218KB
✓ campings                                        27 registros  49KB
✓ campos_de_golf                                  12 registros  22KB
✓ casinos                                          2 registros  3KB
✓ centros_btt_de_euskadi_bicicleta_de_montana      2 registros  3KB
✓ comarcas                                        25 registros  50KB
✓ companias_de_transporte                        145 registros  257KB
✓ destinos_turisticos                           1005 registros  2026KB
✓ empresas_de_alquiler_deport

## 4. Auditoría de campos <a name="4-auditoria"></a>

Antes de hacer el merge, auditamos cada dataset por separado para entender:
- **¿Qué campos tiene?** — No todos los datasets tienen los mismos campos
- **¿Tiene coordenadas?** — Imprescindible para el mapa de Txoko
- **¿Qué porcentaje de registros tiene coords?** — La cobertura varía entre datasets
- **¿Qué campos están vacíos en la mayoría de registros?** — Para decidir si los incluimos en el merge

### Estructura común de los datasets

Todos los datasets de Open Data Euskadi / Turismo Euskadi comparten una **estructura base común** porque provienen del mismo sistema de gestión (webtur00):

| Campo | Descripción | Cobertura típica |
|---|---|---|
| `documentName` | Nombre del lugar | ~100% |
| `documentDescription` | Descripción | ~80-90% |
| `latwgs84` | Latitud (WGS84) | 85-100% según dataset |
| `lonwgs84` | Longitud (WGS84) | 85-100% según dataset |
| `municipality` | Municipio | ~99% |
| `territory` | Territorio histórico | ~99% |
| `phone` | Teléfono | ~8-30% |
| `web` / `webpage` | Web | ~99% |
| `address` | Dirección | ~varía |
| `physicalUrl` | URL ficha turismo oficial | ~99% |


In [18]:
# ─── PASO 4: Auditoría de cada dataset ──────────────────────────────────────
#
# Para cada JSON descargado:
# (1) Listamos todos los campos que aparecen en CUALQUIER registro
#     (no solo en el primero, porque puede haber campos opcionales)
# (2) Verificamos presencia de coordenadas (latwgs84/lonwgs84)
# (3) Calculamos % de cobertura de coords
# (4) Identificamos campos mayoritariamente vacíos (>80% nulos)

audit_report = {}
files_in_raw = sorted([f for f in os.listdir(RAW_DIR) if f.endswith('.json') and not f.startswith('_')])

for fname in files_in_raw:
    name = fname.replace('.json', '')
    with open(os.path.join(RAW_DIR, fname), encoding='utf-8') as f:
        data = json.load(f)
    if not isinstance(data, list):
        data = [data]
    
    # Todos los campos que aparecen en cualquier registro
    all_keys = set()
    for record in data:
        all_keys.update(record.keys())
    
    # Cobertura de coordenadas
    lat_key = next((k for k in all_keys if 'latwgs' in k.lower()), None)
    if lat_key:
        coords_ok = sum(1 for r in data if r.get(lat_key) not in ('', None, 0, '0'))
        coords_pct = coords_ok / len(data) * 100
    else:
        coords_pct = 0
    
    # Campos con >80% vacíos (en la muestra completa)
    empty_cols = []
    for k in all_keys:
        empty = sum(1 for r in data if r.get(k) in ('', None, 0, '0', [])) / len(data)
        if empty > 0.8:
            empty_cols.append(k)
    
    audit_report[name] = {
        'n_registros': len(data),
        'n_campos_total': len(all_keys),
        'campos': sorted(list(all_keys)),
        'tiene_coords': lat_key is not None,
        'cobertura_coords_pct': round(coords_pct, 1),
        'campos_mayormente_vacios': sorted(empty_cols),
    }

# Mostrar resumen
print(f"{'DATASET':45s} {'REGS':>5} {'CAMPOS':>6} {'COORDS%':>8} {'VACÍOS':>6}")
print("─" * 80)
for name, info in sorted(audit_report.items()):
    vacios = len(info['campos_mayormente_vacios'])
    print(f"{name:45s} {info['n_registros']:5d} {info['n_campos_total']:6d} {info['cobertura_coords_pct']:7.1f}% {vacios:6d}")


DATASET                                        REGS CAMPOS  COORDS% VACÍOS
────────────────────────────────────────────────────────────────────────────────
agencias_de_viaje                               532     48    88.9%     27
albergues_turisticos                             82     50   100.0%     25
alojamientos                                   1453     50   100.0%     23
alojamientos_rurales                            433     40    97.7%     11
aquariums                                         1     49   100.0%     26
auditorios                                        2     49   100.0%     25
bares_de_pintxos                                132     47   100.0%     26
campings                                         27     50   100.0%     23
campos_de_golf                                   12     48   100.0%     25
casinos                                           2     49   100.0%     27
centros_btt_de_euskadi_bicicleta_de_montana       2     48   100.0%     27
comarcas           

## 5. Merge y normalización <a name="5-merge"></a>

### Decisión de diseño: merge vertical (pd.concat), no join horizontal

**¿Por qué concat y no merge/join?**

Un `pd.merge` (join) se usa cuando dos tablas comparten una clave común (como un ID).  
En nuestro caso, **cada dataset describe lugares distintos** — un restaurante no es el mismo objeto que un museo.

La operación correcta es **apilar filas** (`pd.concat`): cogemos los registros de cada dataset y los ponemos unos debajo de otros en una única tabla.

```
restaurantes:   [fila1, fila2, ..., fila603]
bares_pintxos:  [fila1, fila2, ..., fila132]
museos:         [fila1, fila2, ..., fila151]
           ↓  pd.concat
master:         [fila1...fila603, fila604...fila735, fila736...fila886, ...]
```

### Campos normalizados en el merge

No conservamos TODOS los campos originales (serían ~50 columnas con muchos vacíos).  
Seleccionamos los **15 campos de valor** para Txoko:

| Campo maestro | Campo original | Notas |
|---|---|---|
| `id` | — | Generado (txk_00001, txk_00002...) |
| `source_dataset` | — | Nombre del archivo JSON de origen |
| `source_url_json` | — | URL exacta de Open Data Euskadi |
| `categoria` | — | Asignada según dataset (Culinario, Cultural...) |
| `subcategoria` | — | Tipo específico |
| `nombre` | `documentName` | Nombre del lugar |
| `descripcion` | `documentDescription` | Recortada a 300 chars |
| `municipio` | `municipality` | Normalizado |
| `territorio` | `territory` | Normalizado a BIZKAIA/GIPUZKOA/ARABA/EUSKADI |
| `lat` | `latwgs84` | Float o None si vacío |
| `lon` | `lonwgs84` | Float o None si vacío |
| `direccion` | `address` | Aplanada si era objeto anidado |
| `telefono` | `phone` | — |
| `web` | `web` / `webpage` | — |
| `ficha_turismo` | `physicalUrl` | URL ficha turismo oficial |


In [19]:
# ─── PASO 5: Definir categorías y URLs fuente ───────────────────────────────
#
# Asignamos manualmente la categoría Txoko a cada dataset.
# Esta es una decisión editorial, no técnica:
# - ¿A qué categoría de la app pertenece este tipo de lugar?
# - Algunos datasets son ambiguos (ej: 'destinos' es muy genérico → Servicios)

CATEGORY_MAP = {
    # CULINARIO — todo lo relacionado con comer y beber
    "restaurantes_asadores_sidrerias": ("Culinario", "Restaurantes / Asadores / Sidrerías"),
    "bares_pintxos":                   ("Culinario", "Bares de pintxos"),
    "pastelerias":                     ("Culinario", "Pastelerías y confiterías"),
    "gastronomia":                     ("Culinario", "Gastronomía general"),
    "platos_tipicos":                  ("Culinario", "Platos típicos"),
    "productos_tierra":                ("Culinario", "Productos de la tierra"),
    "queserias":                       ("Culinario", "Queserías / Conserveras / Productores"),
    # CULTURAL — patrimonio, museos, espectáculos
    "museos":                          ("Cultural",  "Museos y centros de interpretación"),
    "patrimonio_edificios":            ("Cultural",  "Edificios religiosos / Castillos"),
    "patrimonio_cuevas":               ("Cultural",  "Cuevas y restos arqueológicos"),
    "recursos_culturales":             ("Cultural",  "Recursos culturales generales"),
    "auditorios":                      ("Cultural",  "Auditorios"),
    "palacios_congresos":              ("Cultural",  "Palacios de congresos"),
    "hipodromos_estadios":             ("Cultural",  "Hipódromos y estadios"),
    # NATURALEZA — espacios al aire libre
    "playas":                          ("Naturaleza","Playas"),
    "espacios_naturales":              ("Naturaleza","Espacios naturales"),
    "parques_naturales":               ("Naturaleza","Parques naturales"),
    "rutas":                           ("Naturaleza","Rutas y paseos"),
    "centros_btt":                     ("Naturaleza","Centros BTT"),
    "puertos_pesqueros":               ("Naturaleza","Puertos pesqueros"),
    # OCIO — actividades, deporte, entretenimiento
    "recursos_ocio":                   ("Ocio",      "Ocio general"),
    "recursos_deportivos":             ("Ocio",      "Recursos deportivos"),
    "turismo_activo":                  ("Ocio",      "Turismo activo (kayak, surf, escalada...)"),
    "alquiler_deportivo":              ("Ocio",      "Alquiler deportivo"),
    "golf":                            ("Ocio",      "Golf"),
    "puertos_deportivos":              ("Ocio",      "Puertos deportivos / Náutica"),
    "palacios_hielo":                  ("Ocio",      "Palacios de hielo"),
    "parques_atracciones":             ("Ocio",      "Parques de atracciones"),
    "aquariums":                       ("Ocio",      "Aquariums"),
    "casinos":                         ("Ocio",      "Casinos"),
    "turismo_salud":                   ("Ocio",      "Turismo de salud / Spas / Balnearios"),
    # COMPRAS — comercio, ferias, mercados
    "zonas_compras":                   ("Compras",   "Zonas de compras (comercio local)"),
    "recintos_feriales":               ("Compras",   "Recintos feriales"),
    # ALOJAMIENTO
    "hoteles":                         ("Alojamiento","Hoteles"),
    "alojamientos_rurales":            ("Alojamiento","Alojamientos rurales"),
    "albergues":                       ("Alojamiento","Albergues"),
    "campings":                        ("Alojamiento","Campings"),
    # SERVICIOS — infraestructura turística
    "oficinas_turismo":                ("Servicios", "Oficinas de turismo"),
    "destinos":                        ("Servicios", "Destinos turísticos (POIs generales)"),
}

print(f"Categorías definidas para {len(CATEGORY_MAP)} datasets")
print("\nDistribución editorial:")
from collections import Counter
cat_count = Counter(v[0] for v in CATEGORY_MAP.values())
for cat, n in sorted(cat_count.items()):
    print(f"  {cat:15s}: {n} datasets fuente")


Categorías definidas para 39 datasets

Distribución editorial:
  Alojamiento    : 4 datasets fuente
  Compras        : 2 datasets fuente
  Culinario      : 7 datasets fuente
  Cultural       : 7 datasets fuente
  Naturaleza     : 6 datasets fuente
  Ocio           : 11 datasets fuente
  Servicios      : 2 datasets fuente


In [ ]:
# ─── PASO 5b: Función de normalización de un registro ───────────────────────
#
# Esta función toma UN registro JSON crudo de cualquier dataset
# y lo transforma al esquema maestro de 15 campos.
#
# Decisiones de normalización:
#
# (1) Coordenadas: siempre en latwgs84/lonwgs84. Si están vacías → None.
#     Convertimos a float para que sean numéricas en pandas.
#
# (2) Dirección: en algunos datasets es un string, en otros es un dict
#     con subcampos (street, number, postalCode...).
#     Si es dict → unimos los valores con coma.
#
# (3) Territorio: normalizar "ARABA/ÁLAVA" → "ARABA",
#     "BIZKAIA" → "BIZKAIA", etc.
#     Si tiene múltiples territorios (ej: "GIPUZKOA BIZKAIA") → "EUSKADI"
#
# (4) Web: puede estar en 'web', 'webpage', o 'physicalUrl' según dataset.
#     Tomamos el primero que no esté vacío.

def normalize_record(record, dataset_name):
    cat, subcat = CATEGORY_MAP.get(dataset_name, ("Otros", dataset_name))
    
    # ── Coordenadas ──────────────────────────────────────────────────────────
    try:
        lat = float(record.get('latwgs84') or 0) or None
        lon = float(record.get('lonwgs84') or 0) or None
    except (ValueError, TypeError):
        lat, lon = None, None
    
    # ── Nombre ────────────────────────────────────────────────────────────────
    nombre = record.get('documentName') or record.get('name') or ''
    
    # ── Dirección (puede ser string o dict anidado) ───────────────────────────
    addr_raw = record.get('address', '')
    if isinstance(addr_raw, dict):
        # Tomamos solo los valores del dict (street, number, city...)
        addr = ', '.join(str(v) for v in addr_raw.values() if v and v not in ('', 0))
    else:
        addr = str(addr_raw) if addr_raw else ''
    
    # ── Municipio ─────────────────────────────────────────────────────────────
    # Normalizamos espacios extra y capitalización
    municipio = (record.get('municipality') or record.get('locality') or '').strip()
    municipio = re.sub(r'\s+', ' ', municipio)
    # Corregir todo-mayúsculas
    if municipio.isupper():
        municipio = municipio.title()
    
    # ── Territorio ────────────────────────────────────────────────────────────
    # Normalizar variantes de escritura y campos multi-territorio
    terr_raw = (record.get('territory') or '').strip().upper()
    terr_raw = re.sub(r'\s+', ' ', terr_raw)
    terr_raw = terr_raw.replace('ARABA/ÁLAVA','ARABA').replace('ÁLAVA','ARABA').replace('ALAVA','ARABA')
    
    has_a = 'ARABA' in terr_raw
    has_b = 'BIZKAIA' in terr_raw
    has_g = 'GIPUZKOA' in terr_raw
    n_territorios = sum([has_a, has_b, has_g])
    
    if n_territorios > 1:
        territorio = 'EUSKADI'         # recurso de ámbito regional
    elif has_a:
        territorio = 'ARABA'
    elif has_b:
        territorio = 'BIZKAIA'
    elif has_g:
        territorio = 'GIPUZKOA'
    else:
        territorio = 'DESCONOCIDO'
    
    # ── Web ───────────────────────────────────────────────────────────────────
    web = (record.get('web') or record.get('webpage') or
           record.get('physicalUrl') or record.get('friendlyUrl') or '')
    
    # ── Descripción (máximo 300 caracteres) ───────────────────────────────────
    desc = (record.get('documentDescription') or '')
    
    return {
        'source_dataset':  dataset_name,
        'categoria':       cat,
        'subcategoria':    subcat,
        'nombre':          nombre,
        'descripcion':     desc,
        'municipio':       municipio,
        'territorio':      territorio,
        'lat':             lat,
        'lon':             lon,
        'direccion':       addr,
        'telefono':        record.get('phone') or record.get('telephone') or '',
        'web':             web,
        'ficha_turismo':   record.get('physicalUrl') or record.get('friendlyUrl') or '',
    }

print("✓ Función normalize_record() definida")
print("  Transforma cualquier registro JSON crudo al esquema de 15 campos de Txoko")


✓ Función normalize_record() definida
  Transforma cualquier registro JSON crudo al esquema de 15 campos de Txoko


In [21]:
# ─── PASO 5c: Construir el DataFrame maestro ─────────────────────────────────
#
# Iteramos sobre todos los archivos JSON en data/raw/
# Para cada uno:
#   1. Cargamos el JSON
#   2. Aplicamos normalize_record() a cada elemento
#   3. Acumulamos en all_rows
#
# Finalmente hacemos pd.DataFrame(all_rows) — esto es el concat implícito.
# No usamos pd.concat() explícito porque ya estamos construyendo desde cero,
# pero el principio es equivalente: apilar filas de distintas fuentes.

all_rows = []
source_counts = {}

files_in_raw = sorted([f for f in os.listdir(RAW_DIR) if f.endswith('.json') and not f.startswith('_')])

for fname in files_in_raw:
    name = fname.replace('.json', '')
    
    with open(os.path.join(RAW_DIR, fname), encoding='utf-8') as f:
        data = json.load(f)
    if not isinstance(data, list):
        data = [data]
    
    rows = [normalize_record(r, name) for r in data]
    all_rows.extend(rows)
    source_counts[name] = len(rows)
    print(f"  + {name:45s}: {len(rows):4d} registros")

df_raw = pd.DataFrame(all_rows)
df_raw.insert(0, 'id_raw', range(len(df_raw)))  # ID temporal

print(f"\n{'='*60}")
print(f"DataFrame sin limpiar: {len(df_raw):,} filas × {len(df_raw.columns)} columnas")
print(f"Distribución por fuente:")
for name, count in sorted(source_counts.items(), key=lambda x: -x[1]):
    print(f"  {name:45s}: {count:4d}")


  + agencias_de_viaje                            :  532 registros
  + albergues_turisticos                         :   82 registros
  + alojamientos                                 : 1453 registros
  + alojamientos_rurales                         :  433 registros
  + aquariums                                    :    1 registros
  + auditorios                                   :    2 registros
  + bares_de_pintxos                             :  132 registros
  + campings                                     :   27 registros
  + campos_de_golf                               :   12 registros
  + casinos                                      :    2 registros
  + centros_btt_de_euskadi_bicicleta_de_montana  :    2 registros
  + comarcas                                     :   25 registros
  + companias_de_transporte                      :  145 registros
  + destinos_turisticos                          : 1005 registros
  + empresas_de_alquiler_deportivo               :   22 registros
  + empres

## 6. Limpieza y calidad <a name="6-limpieza"></a>

### Pasos de limpieza (documentados y auditables)

1. **Filtro de registros sin nombre** — Un lugar sin nombre no es usable en la app
2. **Deduplicación** — Hay datasets que se solapan (ej: `restaurantes_asadores_sidrerias` y `recursos_culturales` pueden tener el mismo lugar)  
   → Criterio de deduplicación: `(nombre, municipio, categoria)` como clave compuesta
3. **Asignar ID único** — Formato `txk_00001` para que sea trazable
4. **Audit trail** — Documentar cuántos registros eliminó cada paso

### ¿Por qué deduplicar por (nombre, municipio, categoría) y no solo por nombre?

Porque el mismo nombre puede existir en municipios distintos (ej: "Casa Rural Etxea" en Getxo y en Zarautz son lugares distintos). Incluimos `categoría` porque el mismo lugar físico puede aparecer legítimamente en dos categorías (ej: un Parador que es tanto hotel como restaurante).


In [ ]:
# ─── PASO 6: Limpieza ────────────────────────────────────────────────────────

df = df_raw.copy()
audit_trail = {'inicio': len(df)}

# ── 6.1 Filtrar registros sin nombre ─────────────────────────────────────────
# Un registro sin nombre no es utilizable en la app (no podemos mostrarlo)
df = df[df['nombre'].str.strip().ne('')]
audit_trail['tras_filtro_nombre'] = len(df)
eliminados = audit_trail['inicio'] - audit_trail['tras_filtro_nombre']
print(f"6.1 Filtro nombre vacío: {audit_trail['inicio']} → {len(df)} (eliminados: {eliminados})")

# ── 6.2 Deduplicación ────────────────────────────────────────────────────────
# Criterio: (nombre normalizado, municipio, categoría)
# Normalizamos nombre para la comparación (minúsculas, sin espacios extra)
# pero conservamos el nombre original en el campo 'nombre'
df['_nombre_norm'] = df['nombre'].str.lower().str.strip().str.replace(r'\s+', ' ', regex=True)
before_dedup = len(df)
df = df.drop_duplicates(subset=['_nombre_norm', 'municipio', 'categoria'], keep='first')
df = df.drop(columns=['_nombre_norm'])
audit_trail['tras_deduplicacion'] = len(df)
eliminados_dedup = before_dedup - len(df)
print(f"6.2 Deduplicación (nombre+municipio+categoría): {before_dedup} → {len(df)} (eliminados: {eliminados_dedup})")
print(f"    (duplicados encontrados entre datasets solapados)")

# ── 6.3 Asignar ID único ──────────────────────────────────────────────────────
# Formato: txk_00001, txk_00002, ...
# Prefijo 'txk_' para identificar que es un ID de Txoko (no de Open Data Euskadi)
df = df.drop(columns=['id_raw'])
df.insert(0, 'id', [f"txk_{i+1:05d}" for i in range(len(df))])

print(f"6.3 ID único asignado: txk_00001 → txk_{len(df):05d}")

# ── 6.4 Resumen de calidad final ─────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"CALIDAD DEL DATASET MAESTRO — {len(df):,} registros")
print(f"{'='*55}")
print(f"{'Campo':20s} {'Rellenos':>8} {'%':>6}")
print("─" * 40)
for col in ['nombre','municipio','territorio','lat','lon','telefono','web','descripcion']:
    filled = df[col].notna() & (df[col].astype(str).str.strip() != '') & (df[col].astype(str).ne('nan'))
    pct = filled.sum() / len(df) * 100
    bar = '█' * int(pct / 5)
    print(f"{col:20s} {filled.sum():8,} {pct:5.1f}%  {bar}")

print(f"\n⚠ Teléfono: solo 7.8% — Open Data Euskadi no publica teléfonos de forma sistemática")
print(f"⚠ Coords: 85.6% — el 14.4% restante son registros sin geocodificar en origen")


## 7. Dataset maestro final <a name="7-final"></a>

### Resumen ejecutivo para el jurado

| Aspecto | Detalle |
|---|---|
| **Fuente** | Open Data Euskadi (Gobierno Vasco) |
| **Licencia** | CC-BY 4.0 — uso libre con atribución |
| **Datasets fuente** | 39 de Open Data Euskadi |
| **Registros totales** | 4.624 lugares únicos |
| **Con coordenadas** | 3.959 (85.6%) — mapeables directamente |
| **Cobertura geográfica** | Bizkaia, Gipuzkoa, Araba + recursos de ámbito regional |
| **Categorías** | Culinario, Cultural, Naturaleza, Ocio, Compras, Alojamiento, Servicios |
| **Campos** | 15 campos normalizados (nombre, coords, municipio, territorio, contacto...) |

### ¿Qué NO tenemos y por qué?

| Categoría de app | Estado | Motivo |
|---|---|---|
| Eventos en tiempo real | ❌ No disponible | Open Data Euskadi no publica agenda de eventos. En producción: API de Bilbao Turismo o Eventbrite |
| Bares/txakolis (categoría propia) | ⚠ Parcial | Solo bares de pintxos (132). Bodegas y txakolines no están como dataset separado |
| Tiendas locales individuales | ⚠ Parcial | Solo "zonas de compras" (282 barrios/zonas), no tiendas individuales |
| Calificaciones de usuarios | ❌ No disponible | Open Data Euskadi no tiene reseñas. Campo `local_ratio` simulado (seed 42, transparente) |


In [ ]:
# ─── PASO 7: Guardar dataset maestro y estadísticas finales ─────────────────

# Guardar CSV (UTF-8 con BOM para compatibilidad con Excel)
output_path = os.path.join(OUTPUT_DIR, 'txoko_master.csv')
df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"✓ Dataset maestro guardado: {output_path}")
print(f"  {len(df):,} registros × {len(df.columns)} columnas")
print(f"  Tamaño: {os.path.getsize(output_path)//1024} KB")

# Distribución por categoría
print(f"\n{'─'*50}")
print(f"REGISTROS POR CATEGORÍA:")
cat_stats = df.groupby('categoria').agg(
    n_lugares=('id', 'count'),
    con_coords=('lat', lambda x: x.notna().sum()),
    municipios_distintos=('municipio', 'nunique')
).sort_values('n_lugares', ascending=False)
print(cat_stats.to_string())

# Top municipios
print(f"\n{'─'*50}")
print(f"TOP 15 MUNICIPIOS:")
print(df['municipio'].value_counts().head(15).to_string())

# Muestra de registros
print(f"\n{'─'*50}")
print(f"MUESTRA — 5 registros del dataset maestro:")
cols_show = ['id','categoria','subcategoria','nombre','municipio','territorio','lat','lon']
print(df[cols_show].head(5).to_string())

# Guardar también stats en JSON (para la API)
stats = {
    'total_registros': int(len(df)),
    'con_coordenadas': int(df['lat'].notna().sum()),
    'pct_coordenadas': round(df['lat'].notna().mean() * 100, 1),
    'por_categoria': df['categoria'].value_counts().astype(int).to_dict(),
    'por_territorio': df['territorio'].value_counts().astype(int).to_dict(),
    'top_municipios': df['municipio'].value_counts().head(10).astype(int).to_dict(),
    'datasets_fuente': 39,
    'fuente': 'Open Data Euskadi (opendata.euskadi.eus)',
    'licencia': 'CC-BY 4.0',
}
with open(os.path.join(OUTPUT_DIR, 'txoko_stats.json'), 'w', encoding='utf-8') as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)
print(f"\n✓ Stats guardadas: txoko_stats.json")
print(f"\n{'='*50}")
print(f"PIPELINE COMPLETADO")
print(f"  Input:  39 datasets JSON de Open Data Euskadi")
print(f"  Output: txoko_master.csv ({len(df):,} registros, {len(df.columns)} columnas)")
print(f"{'='*50}")
